# Smart Water Management: Leakage-Safe Time-Series Forecasting

This notebook presents a reproducible analysis of prepared Bilancino Lake observations. It forecasts **lake level** and **flow rate** while enforcing chronological validation and preventing future target values from entering the feature set.

The notebook delegates core modeling logic to `src/evaluate.py`, so the interactive analysis and command-line evaluation remain consistent.

## 1. Setup

Run the notebook from the repository root after placing `lakes_fe.csv` in `data/`. See `data/README.md` for the expected schema and source attribution.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from evaluate import TARGETS, build_supervised_frame, evaluate_target, load_data

DATA_PATH = PROJECT_ROOT / 'data' / 'lakes_fe.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results'
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATA_PATH}')

## 2. Load and validate the prepared data

The loader validates the five required columns, rejects missing values, reconstructs the daily date index from June 3, 2002, and checks that dates are unique and increasing.

In [ ]:
data = load_data(DATA_PATH)
overview = pd.DataFrame({
    'value': [len(data), data.index.min().date(), data.index.max().date(), int(data.isna().sum().sum())]
}, index=['rows', 'start', 'end', 'missing_values'])
display(overview)
display(data.head())
display(data.describe().T.round(3))

## 3. Exploratory time-series view

The four measurements show different behavior: temperature is strongly seasonal, rainfall and flow contain sharp events, and lake level is comparatively smoother but still changes across operating periods.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
for axis, column in zip(axes, data.columns):
    axis.plot(data.index, data[column], linewidth=0.8)
    axis.set_ylabel(column)
    axis.grid(alpha=0.25)
axes[-1].set_xlabel('Date')
fig.suptitle('Bilancino Lake prepared daily observations', fontsize=15)
plt.tight_layout()
plt.show()

## 4. Leakage-safe supervised features

To predict time `t`, the pipeline uses only information from `t-1` or earlier:

- Lags of 1, 3, 7, 14, and 28 days for every measurement
- Shifted 7-day and 28-day rolling means and standard deviations
- Sine/cosine annual calendar features

Rolling features are shifted before aggregation. This prevents the current target from appearing inside its own predictors.

In [ ]:
lake_supervised = build_supervised_frame(data, 'Actual_Lake_Level')
print(f'Supervised observations: {len(lake_supervised):,}')
print(f'Predictor count: {lake_supervised.shape[1] - 1}')
display(lake_supervised.head(3))
assert 'Actual_Lake_Level' not in lake_supervised.drop(columns='target').columns
assert lake_supervised.index.is_monotonic_increasing

## 5. Chronological model evaluation

The final 20% of each supervised series is held out. XGBoost is tuned only on the earlier training period with expanding `TimeSeriesSplit` folds. ARIMA(5,1,0), SARIMAX(5,1,0), and a one-step persistence baseline are evaluated on the same test dates.

This cell can take several minutes.

In [ ]:
evaluations = [evaluate_target(data, target, OUTPUT_DIR) for target in TARGETS]
rows = []
for result in evaluations:
    for model in ['persistence', 'xgboost', 'arima_5_1_0', 'sarimax_5_1_0']:
        rows.append({
            'target': result['target'],
            'model': model,
            'rmse': result[model]['rmse'],
            'mae': result[model]['mae'],
        })
results_table = pd.DataFrame(rows)
display(results_table.round(3))

## 6. Forecast comparisons

In [ ]:
for target in TARGETS:
    image_path = OUTPUT_DIR / f'{target.lower()}_forecast.png'
    display(Image(filename=str(image_path)))

## 7. Conclusions

- **Lake level:** XGBoost achieved RMSE 0.900 and MAE 0.683, improving RMSE by about 20.2% over the persistence baseline (RMSE 1.127).
- **Flow rate:** persistence remained the strongest tested method (RMSE 2.085, MAE 0.947). XGBoost did not improve on it, suggesting that abrupt flow events require stronger upstream, precipitation-forecast, or operational-release signals.
- **Methodological result:** removing target leakage produces less dramatic but defensible metrics. Negative results are retained rather than hidden.

Potential extensions include rolling-origin backtesting, probabilistic intervals, raw-data preparation, and additional hydrological drivers.